In [2]:
import pandas as pd
import numpy as np

In [3]:
INPUT_FILE = 'firms_india_sep18_22.csv'
OUTPUT_FILE = 'firms_clean.csv'
df = pd.read_csv(INPUT_FILE)

In [4]:
required_columns = [
    'latitude', 'longitude', 'bright_ti4', 'scan', 'track',
    'acq_date', 'acq_time', 'satellite', 'instrument',
    'confidence', 'version', 'bright_ti5', 'frp', 'daynight'
]
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise SystemExit(f'FIRMS input file does not contain the expected columns: {missing_columns}')

In [5]:
numeric_columns = [
    'latitude', 'longitude', 'bright_ti4', 'scan',
    'track', 'acq_time', 'bright_ti5', 'frp'
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

df['acq_date'] = pd.to_datetime(df['acq_date'], errors='coerce')

In [6]:
df = df[df['acq_date'].notna()].copy()
df = df[
    (df['latitude'] >= 6) & (df['latitude'] <= 37) &
    (df['longitude'] >= 68) & (df['longitude'] <= 98)
].copy()
df = df[df['frp'].notna() & (df['frp'] >= 0)].copy()
df = df.drop_duplicates()

In [7]:
df['year'] = df['acq_date'].dt.year
df['month'] = df['acq_date'].dt.month
df['day'] = df['acq_date'].dt.day
df['day_of_year'] = df['acq_date'].dt.dayofyear
df['day_of_week'] = df['acq_date'].dt.dayofweek
df['log_frp'] = np.log1p(df['frp'])

df = df.sort_values(by=['acq_date', 'latitude', 'longitude']).reset_index(drop=True)

C:\Users\saura\AppData\Local\Temp\ipykernel_22776\2884355957.py:4: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  df['day_of_year'] = df['acq_date'].dt.dayofyear
C:\Users\saura\AppData\Local\Temp\ipykernel_22776\2884355957.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df['day_of_week'] = df['acq_date'].dt.dayofweek


In [8]:
df.to_csv(OUTPUT_FILE, index=False)